# 单因子快速评估 - mom_20d

调用 `factor_eval.run_factor_eval()`，对比同一个因子在全市场、沪深300、中证500、中证1000 的表现。

In [ ]:
from pathlib import Path
import sys
import warnings

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from factor_eval.runner import run_factor_eval
from factor_eval.report import (
    plot_distribution,
    plot_ic,
    plot_ic_decay,
    plot_long_short,
    plot_quantile_returns,
    plot_scorecard,
    plot_yearly_perf,
)
from factor_utils import compute_momentum

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

DUCKDB_PATH = ROOT / "data" / "warehouse" / "ashare.duckdb"
START_DATE = "20240101"
FACTOR_NAME = "mom_20d"
FACTOR_KWARGS = {"window": 20, "col_name": "mom_20d"}
UNIVERSES = ["全市场", "hs300", "csi500", "csi1000"]
UNIVERSE_LABELS = {
    "全市场": "全市场",
    "hs300": "沪深300",
    "csi500": "中证500",
    "csi1000": "中证1000",
}
SHOW_PLOTS = True

## 批量运行

In [ ]:
assert DUCKDB_PATH.exists(), f"DuckDB 文件不存在: {DUCKDB_PATH}"

results = {}
for universe in UNIVERSES:
    print(f"Running {UNIVERSE_LABELS[universe]}...")
    results[universe] = run_factor_eval(
        FACTOR_NAME,
        compute_momentum,
        FACTOR_KWARGS,
        DUCKDB_PATH,
        universe=universe,
        start_date=START_DATE,
    )

pd.DataFrame(
    [
        {
            "universe": UNIVERSE_LABELS[name],
            "n_stocks": result.n_stocks,
            "n_dates": result.n_dates,
            "n_valid_rows": result.n_valid_rows,
            "mean_ic": result.ic_summary["mean_ic"],
            "ic_ir": result.ic_summary["ic_ir"],
        }
        for name, result in results.items()
    ]
).set_index("universe")

## 全市场

In [ ]:
result_all = results["全市场"]
display(plot_scorecard(result_all))
if SHOW_PLOTS:
    display(plot_distribution(result_all))
    display(plot_ic(result_all))
    display(plot_ic_decay(result_all))
    display(plot_quantile_returns(result_all))
    display(plot_long_short(result_all))
    display(plot_yearly_perf(result_all))

## 沪深300

In [ ]:
result_hs300 = results["hs300"]
display(plot_scorecard(result_hs300))
if SHOW_PLOTS:
    display(plot_distribution(result_hs300))
    display(plot_ic(result_hs300))
    display(plot_ic_decay(result_hs300))
    display(plot_quantile_returns(result_hs300))
    display(plot_long_short(result_hs300))
    display(plot_yearly_perf(result_hs300))

## 中证500

In [ ]:
result_csi500 = results["csi500"]
display(plot_scorecard(result_csi500))
if SHOW_PLOTS:
    display(plot_distribution(result_csi500))
    display(plot_ic(result_csi500))
    display(plot_ic_decay(result_csi500))
    display(plot_quantile_returns(result_csi500))
    display(plot_long_short(result_csi500))
    display(plot_yearly_perf(result_csi500))

## 中证1000

In [ ]:
result_csi1000 = results["csi1000"]
display(plot_scorecard(result_csi1000))
if SHOW_PLOTS:
    display(plot_distribution(result_csi1000))
    display(plot_ic(result_csi1000))
    display(plot_ic_decay(result_csi1000))
    display(plot_quantile_returns(result_csi1000))
    display(plot_long_short(result_csi1000))
    display(plot_yearly_perf(result_csi1000))

## 跨市场对比

In [ ]:
def score_value(result, metric):
    scorecard = result.scorecard.set_index("Metric")
    return scorecard.loc[metric, "Value"]


comparison_metrics = {
    "Mean IC": "Mean IC",
    "IC IR": "IC IR",
    "IC Win Rate": "IC Win Rate",
    "Half-life": "IC Half-life",
    "Monotonic": "Q1→Q5 Monotonic",
    "Q5-Q1 Spread": "Q5-Q1 Mean Spread",
    "Mean Turnover": "Mean Turnover",
}
comparison = pd.DataFrame(
    {
        UNIVERSE_LABELS[universe]: [score_value(result, metric) for metric in comparison_metrics.values()]
        for universe, result in results.items()
    },
    index=list(comparison_metrics.keys()),
)
display(comparison)